In [1]:
import numpy as np
import cv2
import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import io
from PIL import Image as PILImage

# ── UTILITIES ───
def pil_to_cv(img: PILImage.Image) -> np.ndarray:
    arr = np.array(img.convert("RGB"))
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

def cv_to_pil(arr: np.ndarray) -> PILImage.Image:
    if arr.ndim == 2:
        return PILImage.fromarray(arr.astype(np.uint8))
    return PILImage.fromarray(cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_BGR2RGB))

def ensure_gray(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 3:
        return cv2.cvtColor(arr, cv2.COLOR_BGR2GRAY)
    return arr

def fig_to_pil(fig: plt.Figure) -> PILImage.Image:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    plt.close(fig)
    return PILImage.open(buf).copy()

def compute_metrics(original, processed):
    o = ensure_gray(original).astype(np.float64)
    p = ensure_gray(processed).astype(np.float64)
    if o.shape != p.shape:
        p = cv2.resize(p, (o.shape[1], o.shape[0]))
    mse = np.mean((o - p) ** 2)
    psnr = 10 * np.log10(255**2 / (mse + 1e-9))
    diff_map = cv2.absdiff(o.astype(np.uint8), p.astype(np.uint8))
    return mse, psnr, diff_map

# ── PHASE 1: NOISE INJECTION ───────
def inject_noise(pil_img, noise_type, intensity):
    img = pil_to_cv(pil_img)
    gray = ensure_gray(img).astype(np.float64)
    h, w = gray.shape
    if noise_type == "Salt & Pepper":
        noisy = gray.copy()
        amount = intensity / 100.0
        noisy[np.random.rand(h, w) < (amount/2)] = 255
        noisy[np.random.rand(h, w) < (amount/2)] = 0
    elif noise_type == "Gaussian":
        noisy = gray + np.random.normal(0, intensity, gray.shape)
    elif noise_type == "Periodic":
        x, y = np.meshgrid(np.arange(w), np.arange(h))
        # Frequency is set to 0.1 which corresponds to roughly U=11, V=11 in FFT
        noisy = gray + intensity * np.sin(0.1 * x + 0.1 * y)
    elif noise_type == "Uniform":
        noisy = gray + np.random.uniform(-intensity, intensity, gray.shape)
    return cv_to_pil(np.clip(noisy, 0, 255).astype(np.uint8))

# ── PHASE 1: SPATIAL & FREQUENCY RESTORATION ────────────────────────────────────
def process_restoration(pil_img, spatial_filt, freq_filt, d0, n_order, nu, nv, nr):
    img = pil_to_cv(pil_img)
    gray = ensure_gray(img).astype(np.float64)
    
    # 1. Apply Spatial Filter First
    if spatial_filt == "Median (Smoothing)":
        gray = cv2.medianBlur(gray.astype(np.uint8), 5).astype(np.float64)
    elif spatial_filt == "Laplacian (Sharpening)":
        lap = cv2.Laplacian(gray, cv2.CV_64F)
        gray = cv2.convertScaleAbs(gray - lap).astype(np.float64)
    elif spatial_filt == "Histogram Equalization":
        gray = cv2.equalizeHist(gray.astype(np.uint8)).astype(np.float64)

    # 2. Apply Frequency Domain Filter
    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    rows, cols = gray.shape
    crow, ccol = rows//2, cols//2
    u_idx, v_idx = np.meshgrid(np.arange(cols)-ccol, np.arange(rows)-crow)
    D = np.sqrt(u_idx**2 + v_idx**2)

    if freq_filt == "Butterworth LP":
        mask = 1 / (1 + (D / d0)**(2 * n_order))
    elif freq_filt == "Butterworth HP":
        mask = 1 / (1 + (d0 / (D + 1e-9))**(2 * n_order))
    elif freq_filt == "Notch Reject":
        mask = np.ones((rows, cols))
        # Target noise spikes at (nu, nv) and symmetric (-nu, -nv)
        for du, dv in [(nu, nv), (-nu, -nv)]:
            Dn = np.sqrt((u_idx - du)**2 + (v_idx - dv)**2)
            mask[Dn <= nr] = 0
    else: mask = np.ones((rows, cols))

    fshift_filtered = fshift * mask
    img_back = np.abs(np.fft.ifft2(np.fft.ifftshift(fshift_filtered)))
    
    mse, psnr, diff = compute_metrics(pil_to_cv(pil_img), img_back)
    
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(np.log(1+np.abs(fshift_filtered)), cmap='hot'); ax[0].set_title("Frequency Spectrum")
    ax[1].imshow(diff, cmap='gray'); ax[1].set_title("Difference Map (Original - Processed)")
    
    return cv_to_pil(img_back), fig_to_pil(fig), f"MSE: {mse:.2f} | PSNR: {psnr:.2f}dB"

# ── PHASE 2: SEGMENTATION & MORPHOLOGY ────────────
def apply_morphology(pil_img, op, k_size, thresh_method):
    img = pil_to_cv(pil_img)
    gray = ensure_gray(img)
    
    if thresh_method == "Otsu":
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    
    kernel = np.ones((k_size, k_size), np.uint8)
    if op == "Erosion": res = cv2.erode(binary, kernel)
    elif op == "Dilation": res = cv2.dilate(binary, kernel)
    elif op == "Opening": res = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    elif op == "Closing": res = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    elif op == "Boundary": res = binary - cv2.erode(binary, kernel)
    elif op == "Skeletonization": res = cv2.ximgproc.thinning(binary)
    else: res = binary
    return cv_to_pil(res)

# ── PHASE 3: FEATURE EXTRACTION & CLASSIFICATION ────────────
_train_features, _train_labels = [], []

def get_feature_vector(binary_img):
    cnts, _ = cv2.findContours(binary_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    peri = cv2.arcLength(c, True)
    rect = cv2.boundingRect(c)
    hull = cv2.convexHull(c)
    solidity = area / cv2.contourArea(hull) if cv2.contourArea(hull) > 0 else 0
    circularity = (4 * np.pi * area) / (peri**2 + 1e-9)
    aspect_ratio = float(rect[2]) / rect[3]
    return [area, peri, circularity, solidity, aspect_ratio]

def train_system(img, label):
    gray = ensure_gray(pil_to_cv(img))
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    feat = get_feature_vector(binary)
    if feat:
        _train_features.append(feat); _train_labels.append(label)
        return f"Successfully added '{label}' to Training Set."
    return "Error: Object not detected."

def run_classification(img):
    if len(_train_features) < 2: return None, "Error: Need at least 2 training classes."
    gray = ensure_gray(pil_to_cv(img))
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    test_feat = get_feature_vector(binary)
    
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(_train_features)
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_sc)
    test_pca = pca.transform(scaler.transform([test_feat]))
    
    prediction = _train_labels[np.argmin([np.linalg.norm(test_pca - tr) for tr in X_pca])]
    
    fig, ax = plt.subplots()
    for lbl in set(_train_labels):
        idx = [i for i, l in enumerate(_train_labels) if l == lbl]
        ax.scatter(X_pca[idx, 0], X_pca[idx, 1], label=lbl, s=100)
    ax.scatter(test_pca[0,0], test_pca[0,1], marker='*', s=250, c='red', label='Test Sample')
    ax.legend(); ax.set_title(f"PCA Classification Result: {prediction}")
    return fig_to_pil(fig), f"The system predicts: {prediction}"

# ── GRADIO INTERFACE ─────────────────────────
with gr.Blocks(title="AI342 Advanced DIPA Pipeline") as demo:
    gr.Markdown("# AI342 Professional DIPA System")
    gr.Markdown("### Restoration → Segmentation → Morphology → PCA Classification")

    with gr.Tab("Phase 1: Restoration & Enhancement"):
        with gr.Row():
            in_img = gr.Image(type="pil", label="Input Image")
            with gr.Column():
                noise_opt = gr.Radio(["Salt & Pepper", "Gaussian", "Periodic", "Uniform"], label="1. Inject Noise Model")
                noise_lvl = gr.Slider(0, 100, 30, label="Noise Intensity")
                noise_btn = gr.Button("Apply Noise")
        
        gr.Markdown("---")
        with gr.Row():
            with gr.Column():
                s_filt = gr.Dropdown(["None", "Median (Smoothing)", "Laplacian (Sharpening)", "Histogram Equalization"], label="2. Spatial Domain")
                f_filt = gr.Dropdown(["None", "Butterworth LP", "Butterworth HP", "Notch Reject"], label="3. Frequency Domain")
                
                gr.Markdown("#### Frequency Controls")
                d0_sld = gr.Slider(1, 200, 30, label="D0 (Cutoff)")
                nu_sld = gr.Slider(-100, 100, 11, label="Notch U (Coordinate)")
                nv_sld = gr.Slider(-100, 100, 11, label="Notch V (Coordinate)")
                nr_sld = gr.Slider(1, 50, 5, label="Notch Radius (R)")
                
                restore_btn = gr.Button("RUN RESTORATION PIPELINE", variant="primary")

        with gr.Row():
            out_img = gr.Image(label="Restored Result")
            out_plot = gr.Image(label="Analysis (Spectrum & Diff Map)")
        metrics_box = gr.Textbox(label="Restoration Metrics (MSE & PSNR)")

        noise_btn.click(inject_noise, [in_img, noise_opt, noise_lvl], in_img)
        restore_btn.click(process_restoration, [in_img, s_filt, f_filt, d0_sld, gr.State(2), nu_sld, nv_sld, nr_sld], [out_img, out_plot, metrics_box])

    with gr.Tab("Phase 2: Segmentation & Morphology"):
        with gr.Row():
            seg_in = gr.Image(type="pil", label="Input for Segmentation")
            with gr.Column():
                t_meth = gr.Radio(["Otsu", "Adaptive"], label="Thresholding Method")
                m_op = gr.Dropdown(["None", "Erosion", "Dilation", "Opening", "Closing", "Boundary", "Skeletonization"], label="Morphological Op")
                m_size = gr.Slider(3, 15, 5, step=2, label="Kernel Size")
                morph_btn = gr.Button("CLEAN MASK", variant="primary")
        seg_out = gr.Image(label="Binary Mask Result")
        morph_btn.click(apply_morphology, [seg_in, m_op, m_size, t_meth], seg_out)

    with gr.Tab("Phase 3: Classification & PCA"):
        with gr.Row():
            train_img = gr.Image(type="pil", label="Training Image")
            label_in = gr.Textbox(label="Object Class Label")
            train_btn = gr.Button("Add to Training Set")
        with gr.Row():
            test_img = gr.Image(type="pil", label="Test Image")
            class_btn = gr.Button("RUN PCA CLASSIFICATION", variant="primary")
        pca_plot = gr.Image(label="PCA Feature Space")
        pred_text = gr.Textbox(label="Classification Report")

        train_btn.click(train_system, [train_img, label_in], pred_text)
        class_btn.click(run_classification, [test_img], [pca_plot, pred_text])

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
